## forced photometry Hello World
**David W. Hogg** (NYU) (MPIA)

### acknowledgement
- I made use of *Claude* for advice and minimal code and documentation help.

### projects and to-dos
- Switch from RA, Dec as input to object id.
- Figure out how to iterate over all possible object ids.
- Make the period fitting fast.

### RSP token notes
*The code below needs an RSP token. Here are Claude's notes about this:*
1. Go to data.lsst.cloud in a browser and log in.
1. Open the user drop-down at the upper right and choose Settings.
1. In the left menu, choose Access tokens.
1. Click Create a token.
1. In the dialog:
1. Name: something like pyvo-tap, named after what you’ll use it with.
1. Scopes: tick read:tap. For image retrieval later, also add read:image.
1. Expiration: set a date, or choose no expiration.
1. Click Create token. The token is shown once, so copy it right away. If you lose it, delete it and make a new one.
2. Save the token with `echo 'RSP_TOKEN=paste-token-here' > .env`

In [ ]:
# !pip install pyvo
# !pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import matplotlib.pyplot as plt
import pyvo, requests

In [ ]:
# step 0: authenticate and connect
load_dotenv()
rsp_token = os.environ["RSP_TOKEN"]
session = requests.Session()
session.headers["Authorization"] = f"Bearer {rsp_token}"
tap = pyvo.dal.TAPService("https://data.lsst.cloud/api/tap", session=session)

In [ ]:
def run(query):
    job = tap.submit_job(query)
    job.run()
    job.wait(phases=["COMPLETED", "ERROR"])
    job.raise_if_error()
    return job.fetch_result().to_table()   # astropy Table

In [ ]:
# step 1: find the objectId at a sky position
ra, dec = 226.7244283498404, -40.30405425755786 # known RR Lyrae
obj = run(f"""
    SELECT objectId, coord_ra, coord_dec
    FROM dp2.Object
    WHERE CONTAINS(POINT('ICRS', coord_ra, coord_dec),
                   CIRCLE('ICRS', {ra}, {dec}, 0.00056)) = 1
""")
object_id = int(obj["objectId"][0])      # expect 743582465276261182

In [ ]:
# step 2: forced photometry for that object, joined to Visit for time
lc = run(f"""
    SELECT fs.band, fs.visit, fs.psfFlux, fs.psfFluxErr,
           fs.psfDiffFlux, fs.psfDiffFluxErr, v.expMidptMJD
    FROM dp2.ForcedSource AS fs
    JOIN dp2.Visit AS v ON fs.visit = v.visit
    WHERE fs.objectId = {object_id}
""")
print(len(lc), "epochs")                # the tutorial reports 887

In [ ]:
# step 3: plot (fluxes in nJy)
fig, ax = plt.subplots(figsize=(7, 4))
for band in "ugrizy":
    m = lc["band"] == band
    if m.any():
        ax.errorbar(lc["expMidptMJD"][m], lc["psfFlux"][m],
                    yerr=lc["psfFluxErr"][m], fmt=".", label=band, alpha=0.7)
ax.set_xlabel("MJD"); ax.set_ylabel("psfFlux [nJy]"); ax.legend(ncol=3)
plt.show()

In [ ]:
# step 4: do slow 6-band periodogram
